# 單物件分類與定位

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/04-localization/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.4.1'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

### 可選：40 步學習實驗

跑完下面的完整實驗後，另開一個 code cell 執行：

```python
!python scripts/run_learning_extensions.py --section 04-localization
from IPython.display import SVG, display
display(SVG(filename='artifacts/runs/learning/04-localization/learning.svg'))
```

資料、optimizer 與結果的限制見網頁；這是另外的補充實驗，不取代下面的完整實驗。

In [1]:
"""One object/image, separate class/box heads; three-step CPU smoke test."""
from pathlib import Path
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/miniyolo-matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/miniyolo-cache")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import torch
from torch import nn


def to_xyxy(boxes):
    center, size = boxes[..., :2], boxes[..., 2:]
    return torch.cat((center - size / 2, center + size / 2), -1)


class Localizer(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
        self.class_head = nn.Linear(4, 2)
        self.box_head = nn.Linear(4 * 16 * 16, 4)

    def forward(self, images):
        features = self.body(images)
        return self.class_head(features.mean((2, 3))), self.box_head(features.flatten(1)).sigmoid()


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    left, right = torch.zeros(1, 1, 4, 4), torch.zeros(1, 1, 4, 4)
    left[0, 0, 1, 0], right[0, 0, 1, 3] = 1, 1
    assert torch.equal(left.mean((2, 3)), right.mean((2, 3)))
    print(f"two different positions -> same global mean={left.mean().item():.4f}")
    images = torch.zeros(2, 3, 32, 32)
    pixel_boxes = torch.tensor([[4., 6., 16., 18.], [16., 10., 28., 22.]])
    labels = torch.tensor([0, 1])
    for i, box in enumerate(pixel_boxes.long()):
        x1, y1, x2, y2 = box.tolist()
        images[i, 0 if i == 0 else 2, y1:y2, x1:x2] = 1
    centers = (pixel_boxes[:, :2] + pixel_boxes[:, 2:]) / 2
    sizes = pixel_boxes[:, 2:] - pixel_boxes[:, :2]
    targets = torch.cat((centers, sizes), 1) / 32
    assert torch.allclose(targets[0], torch.tensor([.3125, .375, .375, .375]))
    model = Localizer()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    before = model.box_head.weight.detach().clone()
    model.train()
    for step in range(3):
        optimizer.zero_grad(set_to_none=True)
        logits, boxes = model(images)
        class_loss = nn.functional.cross_entropy(logits, labels)
        box_loss = nn.functional.mse_loss(boxes, targets)
        loss = class_loss + 5 * box_loss
        loss.backward()
        assert model.box_head.weight.grad.abs().sum() > 0
        optimizer.step()
        print(f"step={step}, classification={class_loss.item():.4f}, "
              f"box={box_loss.item():.4f}, total={loss.item():.4f}")
    assert not torch.equal(before, model.box_head.weight)
    model.eval()
    with torch.no_grad():
        logits, normalized_boxes = model(images)
        predicted_boxes = to_xyxy(normalized_boxes) * 32
    assert logits.shape == (2, 2) and normalized_boxes.shape == (2, 4)
    print(f"first_target_cxcywh={targets[0].tolist()}, class_shape={tuple(logits.shape)}, box_shape={tuple(normalized_boxes.shape)}")
    print(f"predicted pixel xyxy={predicted_boxes.tolist()}")
    fig, axes = plt.subplots(1, 2, figsize=(7, 4), constrained_layout=True)
    for i, ax in enumerate(axes):
        ax.imshow(images[i].permute(1, 2, 0).numpy(), extent=(0, 32, 32, 0))
        for box, color, label in ((pixel_boxes[i], "lime", "GT"), (predicted_boxes[i], "gold", "pred")):
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor=color, label=label))
        ax.set_xlim(0, 32)
        ax.set_ylim(32, 0)
        ax.legend()
        ax.set_title("Synthetic image; only 3 steps")
    path = Path("artifacts/04-localization.png")
    path.parent.mkdir(exist_ok=True)
    fig.savefig(path, dpi=130)
    plt.close(fig)
    print(f"overlay={path}; no held-out detection claim")


if __name__ == "__main__":
    main()


two different positions -> same global mean=0.0625
step=0, classification=0.7173, box=0.0185, total=0.8099
step=1, classification=0.7146, box=0.0127, total=0.7779
step=2, classification=0.7123, box=0.0091, total=0.7576
first_target_cxcywh=[0.3125, 0.375, 0.375, 0.375], class_shape=(2, 2), box_shape=(2, 4)
predicted pixel xyxy=[[7.488578796386719, 7.199568271636963, 20.595396041870117, 19.995590209960938], [9.941701889038086, 8.684192657470703, 23.892370223999023, 22.775699615478516]]
overlay=artifacts/04-localization.png; no held-out detection claim
